# Heterogeneous-Oracle World-Probing for NL-to-FOL Faithfulness on FOLIO

This notebook demonstrates the **four-phase heterogeneous-oracle world-probing pipeline** for evaluating how faithfully large language models translate natural language (NL) into first-order logic (FOL) on the [FOLIO](https://huggingface.co/datasets/yale-nlp/folio) validation dataset.

**Pipeline overview:**
- **Phase 0 (Beam Recall Gate):** Generate k=5 FOL candidate translations per example using Llama-3.1-8B and check whether any candidate matches the gold label via Z3. This gates whether the generator is good enough to proceed.
- **Phase 1 (Oracle Pilot):** Calibrate the Qwen-2.5-7B oracle on 30 hand-crafted (sentence, world, truth) triples. The oracle must achieve ≥72% accuracy.
- **Phase 2 (Main Pipeline):** For each of 204 FOLIO examples, generate k=5 FOL candidates, then generate m=8 "diagnostic worlds" — small closed-world interpretations designed to discriminate between candidate translations. Score each candidate by how well its FOL truth value (computed deterministically) agrees with the oracle's natural-language truth judgment in those worlds. Select the top-scoring candidate and evaluate downstream via Z3.
- **Phase 3 (Baselines & Ablations):** Compare against top-1 selection, self-consistency clustering, direct LLM judge, same-model oracle, random worlds, and m=4 worlds.

**Key finding:** Low beam recall (38%) indicates Llama-3.1-8B generates noisy FOL for complex multi-premise FOLIO examples, creating a ceiling that limits all selection-based methods.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT pre-installed on Colab
_pip('loguru==0.7.2')

# Core packages — pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('matplotlib==3.10.0')

In [ ]:
import json
import math
import os
import re
import random
import sys
from collections import Counter
from typing import Optional

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from loguru import logger
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-8ad782-beam-recall-as-the-binding-constraint-an/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded data with {len(data['datasets'][0]['examples'])} examples")

## Config

All tunable parameters are defined here. The demo uses the minimum values from the original experiment. The original (full-run) values are shown in comments.

In [ ]:
GENERATOR_MODEL = "meta-llama/llama-3.1-8b-instruct"
ORACLE_MODEL_PRIMARY = "qwen/qwen-2.5-7b-instruct"
ORACLE_MODEL_FALLBACK = "qwen/qwen-2.5-72b-instruct"

CANDIDATE_K = 2      # original: 5  — k FOL candidates generated per example
WORLD_M = 2          # original: 8  — m diagnostic worlds generated per candidate set
N_PHASE0 = 3         # original: 50 — examples sampled for beam recall gate
N_STRATIFY = 3       # original: 204 (all FOLIO validation) — examples for main pipeline

## Helper Functions

These are the pure-Python helper functions from `method.py` that do not require LLM API calls. They handle label normalization, FOL response parsing, Jaccard similarity, and metrics computation.

In [ ]:
# ── Label normalization ───────────────────────────────────────────────────────

_LABEL_MAP = {
    "true": "Entailment",
    "entailment": "Entailment",
    "false": "Contradiction",
    "contradiction": "Contradiction",
    "uncertain": "Uncertain",
    "neutral": "Uncertain",
    "unknown": "Uncertain",
}


def _normalize_label(label: str) -> str:
    return _LABEL_MAP.get(label.strip().lower(), "Uncertain")


# ── FOL Parsing from LLM output ──────────────────────────────────────────────

def parse_fol_response(text: str) -> Optional[dict]:
    """Extract premises_fol list and conclusion_fol from LLM response."""
    premises = []
    conclusion = None

    # Look for labeled lines
    lines = text.strip().split("\n")
    for line in lines:
        line = line.strip()
        if not line:
            continue
        # Premise N: formula
        pm = re.match(r"(?:premise\s*\d+\s*:|p\d+\s*:)\s*(.+)", line, re.IGNORECASE)
        if pm:
            premises.append(pm.group(1).strip())
            continue
        # Conclusion: formula
        cm = re.match(r"(?:conclusion\s*:)\s*(.+)", line, re.IGNORECASE)
        if cm:
            conclusion = cm.group(1).strip()
            continue

    # Fallback: lines that look like FOL
    if not premises and not conclusion:
        fol_lines = []
        for line in lines:
            line = line.strip()
            if any(kw in line for kw in ("forall", "exists", "->", " & ", " | ", "not ")):
                if ":" in line:
                    line = line.split(":", 1)[1].strip()
                fol_lines.append(line)
        if len(fol_lines) >= 2:
            premises = fol_lines[:-1]
            conclusion = fol_lines[-1]
        elif len(fol_lines) == 1:
            conclusion = fol_lines[0]

    if not conclusion and premises:
        conclusion = premises.pop()

    if not conclusion:
        return None

    return {"premises_fol": premises, "conclusion_fol": conclusion}


# ── Jaccard similarity (used in self-consistency baseline) ───────────────────

def _jaccard(a: str, b: str) -> float:
    ta = set(a.lower().split())
    tb = set(b.lower().split())
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)


# ── Metrics ──────────────────────────────────────────────────────────────────

def compute_accuracy(results: list[dict]) -> float:
    if not results:
        return 0.0
    return sum(1 for r in results if r.get("correct")) / len(results)


def stratified_accuracy(results: list[dict]) -> dict[str, float]:
    by_label: dict[str, list] = {}
    for r in results:
        by_label.setdefault(r.get("label", "Uncertain"), []).append(r.get("correct", False))
    return {label: sum(v) / len(v) for label, v in by_label.items()}


def compute_confidence_interval(results: list[dict]) -> tuple[float, float]:
    n = len(results)
    if n == 0:
        return (0.0, 0.0)
    p = compute_accuracy(results)
    se = math.sqrt(p * (1 - p) / n)
    return (max(0.0, p - 1.96 * se), min(1.0, p + 1.96 * se))

## Load Pre-computed Results

The full pipeline (Phases 0–3) requires LLM API calls to OpenRouter (Llama-3.1-8B generator + Qwen-2.5-7B oracle) and Z3 for formal inference. Those results are pre-computed and stored in `mini_demo_data.json`.

We extract the per-example predictions for all 7 conditions (main method + 3 baselines + 3 ablations) from the loaded data.

In [ ]:
examples_raw = data["datasets"][0]["examples"]
metadata = data["metadata"]
summary = metadata["summary"]

# Reconstruct per-example result dicts for each condition
def _correct(gold, pred):
    return bool(pred) and pred.strip().lower() == gold.strip().lower()

main_results = []
top1_results = []
sc_results = []
judge_results = []
ablation_a = []
ablation_b = []
ablation_c = []

for i, ex in enumerate(examples_raw):
    gold = ex["output"]
    main_results.append({"example_id": i, "label": gold, "verdict": ex["predict_main_method"],     "correct": _correct(gold, ex["predict_main_method"])})
    top1_results.append({"example_id": i, "label": gold, "verdict": ex["predict_top1_baseline"],   "correct": _correct(gold, ex["predict_top1_baseline"])})
    sc_results.append(  {"example_id": i, "label": gold, "verdict": ex["predict_self_consistency"],"correct": _correct(gold, ex["predict_self_consistency"])})
    judge_results.append({"example_id": i, "label": gold, "verdict": ex["predict_direct_judge"],   "correct": _correct(gold, ex["predict_direct_judge"])})
    ablation_a.append(  {"example_id": i, "label": gold, "verdict": ex["predict_ablation_same_oracle"],   "correct": _correct(gold, ex["predict_ablation_same_oracle"])})
    ablation_b.append(  {"example_id": i, "label": gold, "verdict": ex["predict_ablation_random_worlds"], "correct": _correct(gold, ex["predict_ablation_random_worlds"])})
    ablation_c.append(  {"example_id": i, "label": gold, "verdict": ex["predict_ablation_m4"],     "correct": _correct(gold, ex["predict_ablation_m4"])})

print(f"Loaded {len(examples_raw)} examples across 7 conditions")
print(f"Labels: {[ex['output'] for ex in examples_raw]}")

## Phase 0: Beam Recall Gate

Phase 0 checks whether the generator model (Llama-3.1-8B) can produce at least one correct FOL translation among k=5 candidates for a random sample of N_PHASE0 examples. A recall below 40% triggers a fallback to a larger generator.

The summary below comes from the full 204-example run stored in the data file.

In [ ]:
beam_recall = summary["phase0"]["beam_recall"]
n_phase0 = summary["phase0"]["n"]

logger.info(f"=== Phase 0: Beam Recall Gate (n={n_phase0}) ===")
logger.info(f"beam_recall={beam_recall:.3f}")

if beam_recall < 0.40:
    logger.warning(
        f"Beam recall {beam_recall:.3f} < 0.40 gate. "
        "Switching generator to llama-3.3-70b-instruct for re-check."
    )
else:
    logger.info("Phase 0 gate passed.")

print(f"\nPhase 0 result: beam_recall = {beam_recall:.1%}  (n={n_phase0})")
print(f"Gate threshold: 60% (target) / 40% (hard stop)")
print(f"Outcome: {'BELOW target — low beam recall is the binding constraint' if beam_recall < 0.60 else 'PASSED'}")

## Phase 1: Oracle Pilot Calibration

Phase 1 calibrates the oracle model (Qwen-2.5-7B) on 30 hand-crafted (sentence, world, truth) triples spanning three categories: simple predication, negation, and quantifier scope. The oracle must achieve ≥72% accuracy to be trusted as a truth judge in Phase 2.

Below we show the PILOT_TRIPLES data structure and the calibration result from the full run.

In [ ]:
# Sample of the pilot triples (first 3 from each category for illustration)
PILOT_TRIPLES_SAMPLE = [
    # Simple positive predication
    {"sentence": "Alice is a student.", "world": {"domain": ["alice", "bob"], "atoms": {"Student(alice)": True, "Student(bob)": False}}, "ground_truth": True},
    {"sentence": "Bob is not a student.", "world": {"domain": ["alice", "bob"], "atoms": {"Student(alice)": True, "Student(bob)": False}}, "ground_truth": True},
    {"sentence": "Alice is a doctor.", "world": {"domain": ["alice"], "atoms": {"Doctor(alice)": False}}, "ground_truth": False},
    # Negation
    {"sentence": "No cats are dogs.", "world": {"domain": ["felix"], "atoms": {"Cat(felix)": True, "Dog(felix)": False}}, "ground_truth": True},
    {"sentence": "No cats are dogs.", "world": {"domain": ["rex"], "atoms": {"Cat(rex)": True, "Dog(rex)": True}}, "ground_truth": False},
    # Quantifier scope
    {"sentence": "Every student likes some teacher.", "world": {"domain": ["s1", "t1"], "atoms": {"Student(s1)": True, "Teacher(t1)": True, "Likes(s1, t1)": True}}, "ground_truth": True},
]

pilot_accuracy = summary["phase1"]["oracle_accuracy"]
oracle_model = summary["phase1"]["oracle_model"]

logger.info(f"=== Phase 1: Oracle Pilot (n={summary['phase1']['n']}) model={oracle_model} ===")
logger.info(f"Oracle accuracy: {pilot_accuracy:.3f}")

print(f"\nPhase 1 result: oracle_accuracy = {pilot_accuracy:.1%}  (n={summary['phase1']['n']})")
print(f"Gate threshold: 72%")
print(f"Outcome: {'PASSED' if pilot_accuracy >= 0.72 else 'FAILED'}")
print(f"Oracle model used: {oracle_model}")
print(f"\nSample pilot triple:")
t = PILOT_TRIPLES_SAMPLE[0]
print(f"  Sentence: '{t['sentence']}'")
print(f"  World domain: {t['world']['domain']}")
print(f"  World atoms: {t['world']['atoms']}")
print(f"  Ground truth: {t['ground_truth']}")

## Phase 2: Main Pipeline — World Scoring Logic

The key innovation of Phase 2 is **world-probing**: for each example, m=WORLD_M "diagnostic worlds" are generated by the oracle model. A candidate FOL translation is scored by how often its deterministic truth value (computed by `eval_formula`) agrees with the oracle's natural-language truth judgment in each world.

Below we inspect the pre-computed world scores for the demo examples and identify which candidate was selected.

In [ ]:
logger.info(f"=== Phase 2: Main Pipeline (n={len(examples_raw[:N_STRATIFY])}, m={WORLD_M}, oracle={oracle_model}) ===")

# Inspect the world scores and candidate selection for each demo example
for i, ex in enumerate(examples_raw[:N_STRATIFY]):
    scores = json.loads(ex["metadata_world_scores"])
    best_idx = max(range(len(scores)), key=lambda j: scores[j]) if scores else 0
    gold = ex["output"]
    verdict = ex["predict_main_method"]
    correct = _correct(gold, verdict)

    print(f"\nExample {i}: gold={gold}")
    print(f"  Conclusion: {ex['input'].split(chr(10))[-1][:80]}")
    print(f"  World scores (k candidates): {scores}")
    print(f"  Best candidate index: {best_idx}  (score={scores[best_idx]:.3f})")
    print(f"  Verdict: {verdict!r:15s}  correct={correct}")

## Phase 3: Baselines

Three baselines reuse the same candidate cache from Phase 2 (zero additional API cost):
- **Top-1**: Select the first candidate (equivalent to greedy/temperature=0 generation)
- **Self-consistency**: Jaccard-cluster the k conclusion_fol strings; pick the centroid of the largest cluster
- **Direct judge**: Ask the oracle to select the best candidate directly by number

In [ ]:
logger.info("=== Phase 3: Baselines ===")

# Demonstrate Jaccard self-consistency clustering on the loaded candidates
for i, ex in enumerate(examples_raw[:N_STRATIFY]):
    # Parse the stored candidate strings for illustration
    try:
        cands = json.loads(ex["metadata_candidates"])
    except Exception:
        cands = []

    formulas = [c["conclusion_fol"] for c in cands if c and c.get("conclusion_fol")]

    if len(formulas) >= 2:
        # Pairwise Jaccard similarity between conclusion_fol tokens
        sim_matrix = [[_jaccard(formulas[a], formulas[b]) for b in range(len(formulas))]
                      for a in range(len(formulas))]
        print(f"\nExample {i} — {len(formulas)} valid FOL conclusions")
        print(f"  Jaccard sim[0,1] = {sim_matrix[0][1]:.3f}")
        print(f"  Top-1 verdict: {ex['predict_top1_baseline']!r}")
        print(f"  Self-consistency verdict: {ex['predict_self_consistency']!r}")
    else:
        print(f"\nExample {i} — fewer than 2 parseable candidates")

## Metrics Computation

We compute overall accuracy and 95% confidence intervals for all conditions using the pre-computed results. The metrics use the same `compute_accuracy`, `compute_confidence_interval`, and `stratified_accuracy` functions from `method.py`.

In [ ]:
main_acc  = compute_accuracy(main_results)
top1_acc  = compute_accuracy(top1_results)
sc_acc    = compute_accuracy(sc_results)
judge_acc = compute_accuracy(judge_results)
aa_acc    = compute_accuracy(ablation_a)
ab_acc    = compute_accuracy(ablation_b)
ac_acc    = compute_accuracy(ablation_c)

main_ci   = compute_confidence_interval(main_results)
by_label  = stratified_accuracy(main_results)

print("=== Accuracy by condition (demo subset) ===")
print(f"Main method (world-probing)  : {main_acc:.1%}  95% CI [{main_ci[0]:.1%}, {main_ci[1]:.1%}]")
print(f"--- Baselines ---")
print(f"Top-1                        : {top1_acc:.1%}")
print(f"Self-consistency             : {sc_acc:.1%}")
print(f"Direct judge                 : {judge_acc:.1%}")
print(f"--- Ablations ---")
print(f"Same-model oracle (Llama)    : {aa_acc:.1%}")
print(f"Random worlds                : {ab_acc:.1%}")
print(f"m=4 worlds                   : {ac_acc:.1%}")

print(f"\n=== Accuracy by label (main method, demo subset) ===")
for lbl, acc in by_label.items():
    print(f"  {lbl:<15}: {acc:.1%}")

print(f"\n=== Full-run summary (from metadata) ===")
print(f"  beam_recall   : {summary['phase0']['beam_recall']:.1%}  (n={summary['phase0']['n']})")
print(f"  oracle_acc    : {summary['phase1']['oracle_accuracy']:.1%}  (n={summary['phase1']['n']})")
print(f"  main_method   : {summary['main_method']['accuracy']:.1%}  (n={summary['main_method']['n']})")
print(f"  top1_baseline : {summary['baselines']['top1']['accuracy']:.1%}")

## Results Visualization

Two charts summarize the experiment:
1. **Accuracy comparison** — main method vs all baselines and ablations (full-run, n=204)
2. **By-label breakdown** — accuracy split by Entailment / Contradiction / Uncertain for the main method

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Chart 1: Accuracy by condition (full-run numbers from metadata) ──────────
s = summary
conditions = [
    ("Main method\n(world-probing)", s["main_method"]["accuracy"], "#2196F3"),
    ("Top-1\nbaseline",              s["baselines"]["top1"]["accuracy"], "#9E9E9E"),
    ("Self-consistency\nbaseline",   s["baselines"]["self_consistency"]["accuracy"], "#9E9E9E"),
    ("Direct judge\nbaseline",       s["baselines"]["direct_judge"]["accuracy"], "#9E9E9E"),
    ("Same-model\noracle ablation",  s["ablations"]["same_model_oracle"]["accuracy"], "#FF9800"),
    ("Random worlds\nablation",      s["ablations"]["random_worlds"]["accuracy"], "#FF9800"),
    ("m=4 worlds\nablation",         s["ablations"]["m4"]["accuracy"], "#FF9800"),
]
labels, accs, colors = zip(*conditions)

ax = axes[0]
bars = ax.bar(range(len(labels)), accs, color=colors, edgecolor="white", linewidth=0.5)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylim(0, 0.4)
ax.set_ylabel("Accuracy")
ax.set_title(f"Accuracy by Condition (n=204 FOLIO validation)")
ax.axhline(s["main_method"]["accuracy"], color="#2196F3", linestyle="--", alpha=0.4, linewidth=1)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 0.005, f"{acc:.1%}", ha="center", va="bottom", fontsize=8)

legend_patches = [
    mpatches.Patch(color="#2196F3", label="Main method"),
    mpatches.Patch(color="#9E9E9E", label="Baselines"),
    mpatches.Patch(color="#FF9800", label="Ablations"),
]
ax.legend(handles=legend_patches, fontsize=8)

# ── Chart 2: By-label accuracy for main method ───────────────────────────────
by_lbl = s["main_method"]["by_label"]
label_names = list(by_lbl.keys())
label_accs  = [by_lbl[l] for l in label_names]
label_colors = {"Entailment": "#4CAF50", "Contradiction": "#F44336", "Uncertain": "#FF9800"}
lcolors = [label_colors.get(l, "#9E9E9E") for l in label_names]

ax2 = axes[1]
bars2 = ax2.bar(label_names, label_accs, color=lcolors, edgecolor="white", linewidth=0.5)
ax2.set_ylim(0, 0.5)
ax2.set_ylabel("Accuracy")
ax2.set_title("Main Method: Accuracy by Label (n=204)")
for bar, acc in zip(bars2, label_accs):
    ax2.text(bar.get_x() + bar.get_width()/2, acc + 0.01, f"{acc:.1%}", ha="center", va="bottom", fontsize=10)
ax2.axhline(s["main_method"]["accuracy"], color="black", linestyle="--", alpha=0.5, linewidth=1, label=f"Overall {s['main_method']['accuracy']:.1%}")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig("results_summary.png", dpi=100, bbox_inches="tight")
plt.show()

print("\nKey finding: Beam recall = 38% (well below 60% target).")
print("The generator (Llama-3.1-8B) frequently fails to produce any correct FOL among k=5 candidates,")
print("creating a ceiling that limits all selection-based methods including world-probing.")